# Run a Qwen3.5 reasoning model on a GPU

The Qwen series offers models in different sizes. However, later models
(from the 3.6 or 3.8 series) only come in larger sizes which are not
suitable for consumer GPUs out-of-the-box. Therefore, we will run a
quite new model which was derived from Qwen3.5-9B: 
[XiaomiMiMo/MiMo-V2.6-Distill-Qwen-9B](https://huggingface.co/XiaomiMiMo/MiMo-V2.6-Distill-Qwen-9B)

It has all the features of the original Qwen model, but is much better
in almost any respect.

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [2]:
model_name = "XiaomiMiMo/MiMo-V2.6-Distill-Qwen-9B"

Be sure to always load the model and the tokenizer with the same name. Otherwise, results are completely arbitrary.

In [3]:
# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Check memory usage

In [4]:
!nvidia-smi

Thu Sep 24 16:49:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.91.07              Driver Version: 595.91.07      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        Off |   00000000:26:00.0 Off |                  N/A |
|  0%   43C    P1             77W /  575W |   17714MiB /  32607MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Play with different prompts and `enable_thinking`. You can also modify the system prompt!

In [5]:
# prepare the model input
# prompt = "How many 'i's are in 'inscription'?"
prompt = "How many 'r's are in 'strawberry'?"
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True
)

The text is not yet tokenized, so you can observe the effect of the chat template.

In [6]:
# we check what happens when we remove thinking later
text

"<|im_start|>user\nHow many 'r's are in 'strawberry'?<|im_end|><|im_start|>assistant\n"

As model inputs, we need the `id`s of the tokens.

In [7]:
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

Check the speed of the generation process.

In [13]:
%%time 
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)

CPU times: user 2.34 s, sys: 9.67 ms, total: 2.35 s
Wall time: 2.35 s


The model returns the text including the prompt, so skip our own input.

In [14]:
# only read output, skip input
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

Here we could calculate the number of tokens per second:

In [15]:
len(output_ids)

70

Split the data into the thinking process and the solution.
You can get the `token_id`s from `tokenizer_config.json`.

And this is the solution:

In [16]:
content = tokenizer.decode(output_ids).strip("\n")

In [17]:
print(content)

<think></think>There are 3 'r's in "strawberry".

S-T-R-A-W-B-E-R-R-Y
1.  Position 3: "r"
2.  Position 8: "r"
3.  Position 9: "r"

So, 3 'r's total.<|im_end|>


Most models producs results in *markdown*, which can be converted to HTML:

In [ ]:
from IPython.display import display, Markdown
display(Markdown(content))

## Disable thinking

In [ ]:
# prepare the model input
# prompt = "How many 'i's are in 'inscription'?"
prompt = "How many 'r's are in 'strawberry'?"
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

In [ ]:
text

In [ ]:
%%time 
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)

In [ ]:
# only read output, skip input
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
content = tokenizer.decode(output_ids).strip("\n")
display(Markdown(content))